# Installing and loading libraries (This may take a while)

In [1]:
%%capture
!pip install diffusers accelerate
!pip install torchmetrics[image]
!pip install git+https://github.com/openai/CLIP.git
!pip install stable_baselines3
!pip install shimmy

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt

import skimage.io as io
from PIL import Image

import gym
from gym import spaces
from stable_baselines3 import DQN

from diffusers import StableDiffusionPipeline
from diffusers import DPMSolverMultistepScheduler as DefaultDPMSolver
from diffusers.utils import make_image_grid

from IPython.display import display

from pycocotools.coco import COCO

from functools import partial
import torchvision.transforms as transforms
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.functional.multimodal import clip_score
import clip

# Prep Data

## [TEMP] Import Dataset

In [ ]:
data_dir = "./coco"
ann_url = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
os.makedirs(data_dir, exist_ok=True)

print("Downloading COCO annotations...")
os.system(f"wget {ann_url} -P {data_dir} && unzip {data_dir}/annotations_trainval2017.zip -d {data_dir}")
print("COCO dataset downloaded successfully!")

ann_file='{}/annotations/instances_val2017.json'.format(data_dir)
coco=COCO(ann_file)
ann_file = '{}/annotations/captions_val2017.json'.format(data_dir)
coco_caps=COCO(ann_file)

In [4]:
def random_images(n = 10):
    img_ids = coco.getImgIds()
    imgs = []
    prompts = []
    for i in range(n):
        img = coco.loadImgs(img_ids[np.random.randint(0,len(img_ids))])[0]
        ann_ids = coco_caps.getAnnIds(img_ids=img['id']);
        prompts.append(max([ann['caption'] for ann in coco_caps.loadAnns(ann_ids)], key=len))
        I = io.imread(img['coco_url'])
        imgs.append(Image.fromarray(I))
    return imgs, prompts
real_imgs, prompts = random_images()

## [TEMP] Text Embedding

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

def get_text_embedding(text):
    """
    Converts a text prompt into a CLIP embedding.
    """
    text_tokenized = clip.tokenize([text]).to(device)
    with torch.no_grad():
        text_embedding = model.encode_text(text_tokenized)
    return text_embedding.cpu().numpy().flatten()
text_embeddings = [get_text_embedding(prompt) for prompt in prompts]

# Text-to-Image Model

## Stable Diffusion 1.5

In [6]:
# Add support for setting custom timesteps
class DPMSolverMultistepScheduler(DefaultDPMSolver):
    def set_timesteps(
        self, num_inference_steps=None, device=None,
        timesteps=None
    ):
        if timesteps is None:
            super().set_timesteps(num_inference_steps, device)
            return

        all_sigmas = np.array(((1 - self.alphas_cumprod) / self.alphas_cumprod) ** 0.5)
        self.sigmas = torch.from_numpy(all_sigmas[timesteps])
        self.timesteps = torch.tensor(timesteps[:-1]).to(device=device, dtype=torch.int64) # Ignore the last 0

        self.num_inference_steps = len(timesteps)

        self.model_outputs = [
            None,
        ] * self.config.solver_order
        self.lower_order_nums = 0

        # add an index counter for schedulers that allow duplicated timesteps
        self._step_index = None
        self._begin_index = None
        self.sigmas = self.sigmas.to("cpu")  # to avoid too much CPU/GPU communication

In [ ]:
model_id = "runwayml/stable-diffusion-v1-5"
pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16, variant="fp16").to("cuda")
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
def text_to_image(sampling_schedule, prompt):
    seed = 10
    num_steps = len(sampling_schedule)-1
    torch.manual_seed(seed)
    images = pipe(
        prompt, num_images_per_prompt=1,
        timesteps=sampling_schedule,
    ).images
    return images[0]

## [TODO] SDXL

# RL Framework

In [ ]:
class ScheduleAdjustmentEnv(gym.Env):
    def __init__(self, base_schedule, text_to_image, text_embeddings, real_imgs, prompts):
        super(ScheduleAdjustmentEnv, self).__init__()

        self.base_schedule = base_schedule
        self.text_to_image = text_to_image
        self.text_embeddings = text_embeddings
        self.real_imgs = real_imgs
        self.prompts = prompts

        self.delta_t_ratio = [0.01,0.05,0.1,0.5] #TO BE ADJUSTED
        self.weights = {"fid":1,"clip":1,"step":1} #TO BE ADJUSTED
        self.max_step = 10 #TO BE ADJUSTED
        self.cur_index = 0

        self.generate_all()

        self.max_sample = 10
        self.fid_score_fn = FrechetInceptionDistance(normalize=True)
        self.fid_baseline = self.calculate_fid()

        self.clip_score_fn = partial(clip_score, model_name_or_path="openai/clip-vit-base-patch32")

        # Action space: (index*ratio*increase/decrease)
        self.action_space = spaces.Discrete((len(base_schedule)-2)*len(self.delta_t_ratio)*2)

        # Observation space: sampling_schedule, text_embedding
        self.observation_space = spaces.Dict({
            "sampling_schedule": spaces.Box(low=min(self.base_schedule), high=max(self.base_schedule), shape=(len(base_schedule),), dtype=np.int32),
            "text_embedding": spaces.Box(low=-np.inf, high=np.inf, shape=text_embeddings[0].shape, dtype=np.float32),
        })

    def step(self, action):
        self.steps += 1
        index, op = action//(len(self.delta_t_ratio)*2), action%(len(self.delta_t_ratio)*2)

        # Ensure valid index (ignore t_min and t_max)
        index += 1

        if  op%2==0:
            max_delta_t = self.sampling_schedule[index-1] - self.sampling_schedule[index]
            self.sampling_schedule[index] += int(max_delta_t*self.delta_t_ratio[op//2])
        else:
            max_delta_t = self.sampling_schedule[index] - self.sampling_schedule[index+1]
            self.sampling_schedule[index] -= int(max_delta_t*self.delta_t_ratio[op//2])

        self.generate()
        fid = self.calculate_fid()
        clip = self.calculate_clip()

        reward = self.weights["clip"]*clip/self.clip_baseline - self.weights["fid"]*fid/self.fid_baseline - self.weights["step"]*max((self.steps-self.max_step)/self.max_step,0)
        return {"sampling_schedule":self.sampling_schedule,"text_embedding":self.text_embeddings[self.cur_index]}, reward, False, {}

    def reset(self):
        print("reset")
        self.sampling_schedule = self.base_schedule.copy()
        self.clip_baseline = self.calculate_clip()
        self.fid_baseline = self.calculate_fid(True)
        self.steps = 0
        return {"sampling_schedule":self.sampling_schedule,"text_embedding":self.text_embeddings[self.cur_index]}

    def render(self, mode="human"):
        print("Sampling Schedule:", self.sampling_schedule)

    def set_index(self, i):
        self.cur_index = i

    def generate(self):
        self.gen_imgs[self.cur_index] = self.text_to_image(self.sampling_schedule, self.prompts[self.cur_index])

    def generate_all(self):
        self.gen_imgs = []
        for prompt in self.prompts:
            self.gen_imgs.append(self.text_to_image(self.base_schedule, prompt))
        self.baseline_imgs = self.gen_imgs.copy()

    def calculate_fid(self, baseline = False):
        first_index = max(0,self.cur_index-self.max_sample+1)
        last_index = max(self.max_sample,self.cur_index+1)
        self.fid_score_fn.reset()
        self.fid_score_fn.update(self.preprocess_fid(self.real_imgs[first_index:last_index]), real=True)
        if baseline:
            self.fid_score_fn.update(self.preprocess_fid(self.baseline_imgs[first_index:last_index]), real=False)
        else:
            self.fid_score_fn.update(self.preprocess_fid(self.gen_imgs[first_index:last_index]), real=False)
        return self.fid_score_fn.compute()

    def preprocess_fid(self, images):
        transform = transforms.Compose([
            transforms.Resize((299, 299)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        return torch.cat([transform(img).unsqueeze(0) for img in images])

    def calculate_clip(self):
        clip = self.clip_score_fn(torch.from_numpy(np.array([self.gen_imgs[self.cur_index]])).permute(0, 3, 1, 2), [self.prompts[self.cur_index]]).detach()
        return round(float(clip), 4)


In [ ]:
sampling_schedule = np.array([999, 850, 736, 645, 545, 455, 343, 233, 124, 24, 0])

env = ScheduleAdjustmentEnv(sampling_schedule, text_to_image, text_embeddings, real_imgs, prompts)
obs = env.reset()

In [ ]:
# Train an RL agent (DQN)
num_prompts = 1 #TEMP
#num_prompts = len(prompts)
model = DQN("MultiInputPolicy", env, verbose=1)
for i in range(num_prompts):
    env.set_index(i)
    model.learn(total_timesteps=10)

In [ ]:
# Test trained model
env.set_index(0)
obs = env.reset()
last_reward = -np.inf
for _ in range(10):
    action, _ = model.predict(obs)
    obs, reward, done, _ = env.step(action)
    if reward < last_reward:
        break
    last_reward = reward
    last_obs = obs
    env.render()

In [ ]:
print("Prompt: ",prompts[0])
print("Real Image:")
display(real_imgs[0])
print("Baseline Image:")
display(text_to_image(np.array(sampling_schedule), prompts[0]))
print("New Image:")
display(text_to_image(last_obs['sampling_schedule'], prompts[0]))